In [2]:
import ssl
import certifi
import pandas as pd
import urllib.request
import io

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"

context = ssl.create_default_context(cafile=certifi.where())

with urllib.request.urlopen(url, context=context) as response:
    taxis = pd.read_csv(io.BytesIO(response.read()))



In [3]:
taxis["pickup"] = pd.to_datetime(taxis["pickup"], errors="coerce")
taxis["dropoff"] = pd.to_datetime(taxis["dropoff"], errors="coerce")

taxis["total"] = pd.to_numeric(taxis["total"], errors="coerce")
taxis["tip"] = pd.to_numeric(taxis["tip"], errors="coerce")

print("Nombre de courses :", len(taxis))
display(taxis.head())

Nombre de courses : 6433


,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


In [4]:
taxis["date_course"] = taxis["pickup"].dt.date

activite_par_jour = (
    taxis.groupby("date_course")
    .agg(
        nombre_courses=("pickup", "count"),
        chiffre_affaires=("total", "sum")
    )
    .reset_index()
)

display(activite_par_jour.round(2))

,date_course,nombre_courses,chiffre_affaires
0,2019-02-28,1,6.30
1,2019-03-01,241,4213.83
2,2019-03-02,198,3319.02
3,2019-03-03,169,3027.32
4,2019-03-04,171,3269.08
5,2019-03-05,228,4377.38
6,2019-03-06,257,4673.61
7,2019-03-07,218,4115.70
8,2019-03-08,235,4524.21
9,2019-03-09,204,3785.80


In [5]:
taxis["heure_pickup"] = taxis["pickup"].dt.hour

nombre_courses_par_heure = (
    taxis.groupby("heure_pickup")
    .size()
    .sort_index()
)

heure_plus_chargee = nombre_courses_par_heure.idxmax()
nombre_courses_heure_max = nombre_courses_par_heure.max()

print("Heure la plus chargée :", heure_plus_chargee, "h")
print("Nombre de courses :", nombre_courses_heure_max)

display(nombre_courses_par_heure)

Heure la plus chargée : 18 h
Nombre de courses : 417


heure_pickup
0     205
1     110
2     101
3      67
4      57
5      51
6     142
7     221
8     313
9     319
10    325
11    295
12    334
13    316
14    360
15    330
16    336
17    388
18    417
19    406
20    368
21    355
22    321
23    296
dtype: int64

In [6]:
noms_jours = {
    0: "Lundi",
    1: "Mardi",
    2: "Mercredi",
    3: "Jeudi",
    4: "Vendredi",
    5: "Samedi",
    6: "Dimanche"
}

taxis["jour_semaine"] = taxis["pickup"].dt.dayofweek.map(noms_jours)

pourboire_moyen_par_jour = (
    taxis.groupby("jour_semaine")["tip"]
    .mean()
    .sort_values(ascending=False)
)

jour_pourboire_max = pourboire_moyen_par_jour.idxmax()
montant_pourboire_max = pourboire_moyen_par_jour.max()

print("Jour avec le pourboire moyen le plus élevé :", jour_pourboire_max)
print("Pourboire moyen :", round(montant_pourboire_max, 2), "$")

display(pourboire_moyen_par_jour.round(2))

Jour avec le pourboire moyen le plus élevé : Mercredi
Pourboire moyen : 2.09 $


jour_semaine
Mercredi    2.09
Jeudi       2.06
Lundi       2.03
Mardi       2.00
Vendredi    1.98
Dimanche    1.91
Samedi      1.82
Name: tip, dtype: float64

In [7]:
taxis["duree_course_minutes"] = (
    taxis["dropoff"] - taxis["pickup"]
).dt.total_seconds() / 60

durees_positives = taxis.loc[
    taxis["duree_course_minutes"] > 0,
    "duree_course_minutes"
].dropna()

premier_quartile_duree = durees_positives.quantile(0.25)
troisieme_quartile_duree = durees_positives.quantile(0.75)

ecart_interquartile_duree = (
    troisieme_quartile_duree - premier_quartile_duree
)

borne_inferieure_duree = (
    premier_quartile_duree - 1.5 * ecart_interquartile_duree
)

borne_superieure_duree = (
    troisieme_quartile_duree + 1.5 * ecart_interquartile_duree
)

taxis["duree_aberrante"] = (
    (taxis["duree_course_minutes"] <= 0)
    | (taxis["duree_course_minutes"] < borne_inferieure_duree)
    | (taxis["duree_course_minutes"] > borne_superieure_duree)
)

print("Borne inférieure :", round(borne_inferieure_duree, 2), "minutes")
print("Borne supérieure :", round(borne_superieure_duree, 2), "minutes")
print("Nombre de durées aberrantes :", taxis["duree_aberrante"].sum())

display(
    taxis.loc[
        taxis["duree_aberrante"],
        ["pickup", "dropoff", "duree_course_minutes"]
    ].head(10)
)

Borne inférieure : -11.51 minutes
Borne supérieure : 36.56 minutes
Nombre de durées aberrantes : 364


,pickup,dropoff,duree_course_minutes
56,2019-03-05 17:57:00,2019-03-05 18:45:25,48.416667
111,2019-03-15 10:56:56,2019-03-15 11:37:02,40.100000
127,2019-03-21 18:44:03,2019-03-21 19:30:50,46.783333
128,2019-03-06 19:29:56,2019-03-06 20:08:36,38.666667
151,2019-03-29 10:25:21,2019-03-29 11:14:27,49.100000
162,2019-03-06 17:36:48,2019-03-06 18:43:39,66.850000
188,2019-03-26 15:06:47,2019-03-26 15:59:52,53.083333
218,2019-03-20 11:15:24,2019-03-20 11:59:38,44.233333
225,2019-03-22 15:36:34,2019-03-22 16:34:48,58.233333
250,2019-03-21 10:07:48,2019-03-21 10:52:02,44.233333


In [8]:
taxis["duree_course_minutes_nettoyee"] = (
    taxis["duree_course_minutes"]
)

taxis.loc[
    taxis["duree_aberrante"],
    "duree_course_minutes_nettoyee"
] = float("nan")

duree_moyenne_apres_nettoyage = (
    taxis["duree_course_minutes_nettoyee"].mean()
)

print(
    "Durée moyenne après nettoyage :",
    round(duree_moyenne_apres_nettoyage, 2),
    "minutes"
)

print(
    "Durées manquantes après nettoyage :",
    taxis["duree_course_minutes_nettoyee"].isna().sum()
)

Durée moyenne après nettoyage : 12.32 minutes
Durées manquantes après nettoyage : 364


### Justification des choix

- **Chiffre d'affaires :** j'utilise la colonne `total` pour calculer le montant total facturé par jour.
- **Heure la plus chargée :** je compte les courses selon leur heure de prise en charge.
- **Pourboires :** je compare la moyenne des pourboires selon le jour de la semaine.
- **Durées aberrantes :** je repère les durées nulles, négatives ou atypiques avec la méthode IQR. Je les exclus des calculs de durée, mais conserve les courses afin de ne pas perdre les autres informations.